In [1]:
# =========================
# STEP 2 — DEMAND EDA + EXPORT CLUSTERING INPUT (single cell, independent)
# Input :
#   - /user/data/preprocess/model_ready
# Optional:
#   - /user/data/preprocess/full_panel (để tham chiếu tổng quan)
# Output:
#   - EDA files under /workspace/code/kmeans/demand
#   - clustering-ready long parquet at /user/data/clustering_input/train_series_long
#   - local metadata at /workspace/code/kmeans/clustering_meta
# =========================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# -------------------------
# 1) CONFIG
# -------------------------
INPUT_PATH = "/user/data/preprocess/model_ready"
FULL_PANEL_PATH = "/user/data/preprocess/full_panel"  # optional, chỉ để tham chiếu nếu cần
EDA_OUT_DIR = "/workspace/code/kshape/demand"
CLUSTER_INPUT_OUT = "/user/data/clustering_input/train_series_long"
CLUSTER_META_DIR = "/workspace/code/kshape/clustering_meta"
TAXI_ZONE_LOOKUP_CSV = "/user/data/raw/taxi_zone_lookup.csv"  # optional

TOP_N_DENSE = 5
TOP_N_BORDERLINE = 5
MAX_PLOT_POINTS = 4000

os.makedirs(EDA_OUT_DIR, exist_ok=True)
os.makedirs(CLUSTER_META_DIR, exist_ok=True)

spark = SparkSession.builder.appName("DemandEDAAndClusteringInput").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.parquet.mergeSchema", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")

# -------------------------
# 2) HELPERS
# -------------------------
def ensure_required_columns(df_, required_cols, df_name):
    missing = required_cols - set(df_.columns)
    if missing:
        raise ValueError(f"{df_name} thiếu cột bắt buộc: {sorted(list(missing))}")

def prepare_base_panel(df_):
    ensure_required_columns(df_, {"pickup_bin_30m", "PULocationID", "pickup_demand"}, "Input parquet")
    out = (
        df_.select(
            F.col("pickup_bin_30m").cast("timestamp").alias("pickup_bin_30m"),
            F.col("PULocationID").cast("int").alias("PULocationID"),
            F.col("pickup_demand").cast("double").alias("pickup_demand"),
            *[F.col(c) for c in df_.columns if c not in {"pickup_bin_30m", "PULocationID", "pickup_demand"}]
        )
        .filter(F.col("pickup_bin_30m").isNotNull())
        .filter(F.col("PULocationID").isNotNull())
        .withColumn("pickup_demand", F.coalesce(F.col("pickup_demand"), F.lit(0.0)).cast("double"))
    )

    if "hour" not in out.columns:
        out = out.withColumn("hour", F.hour("pickup_bin_30m").cast("int"))
    if "day_of_week" not in out.columns:
        out = out.withColumn("day_of_week", ((F.dayofweek("pickup_bin_30m") + 5) % 7).cast("int"))
    return out

def choose_train_scope(df_):
    if "dataset_split" in df_.columns:
        return df_.filter(F.col("dataset_split") == "train"), "dataset_split=train"
    if "split" in df_.columns:
        return df_.filter(F.col("split") == "train"), "split=train"
    return df_, "all_rows"

def build_location_stats(df_):
    return (
        df_.groupBy("PULocationID")
           .agg(
               F.count("*").alias("n_bins"),
               F.sum(F.when(F.col("pickup_demand") > 0, 1).otherwise(0)).alias("n_active_bins"),
               F.sum(F.when(F.col("pickup_demand") == 0, 1).otherwise(0)).alias("n_zero_bins"),
               F.avg("pickup_demand").alias("avg_pickup_demand"),
               F.stddev("pickup_demand").alias("std_pickup_demand"),
               F.max("pickup_demand").alias("max_pickup_demand"),
               F.expr("percentile_approx(pickup_demand, 0.5)").alias("median_pickup_demand"),
               F.expr("percentile_approx(pickup_demand, 0.95)").alias("p95_pickup_demand"),
               F.expr("percentile_approx(pickup_demand, 0.99)").alias("p99_pickup_demand")
           )
           .withColumn("active_ratio", F.col("n_active_bins") / F.col("n_bins"))
           .withColumn("zero_ratio", F.col("n_zero_bins") / F.col("n_bins"))
           .withColumn(
               "cv_pickup_demand",
               F.when(F.col("avg_pickup_demand") > 0, F.col("std_pickup_demand") / F.col("avg_pickup_demand"))
                .otherwise(None)
           )
    )

def downsample_pdf(pdf, max_points=4000):
    if len(pdf) <= max_points:
        return pdf
    step = int(np.ceil(len(pdf) / max_points))
    return pdf.iloc[::step].copy()

def atomic_local_csv(df_pd, path):
    tmp = path + ".tmp"
    df_pd.to_csv(tmp, index=False)
    os.replace(tmp, path)

# -------------------------
# 3) READ INPUT
# -------------------------
df_raw = spark.read.parquet(INPUT_PATH)
df_raw = prepare_base_panel(df_raw)
df_train, scope_name = choose_train_scope(df_raw)

print("=" * 120)
print("STEP 2 — DEMAND EDA + EXPORT CLUSTERING INPUT")
print("=" * 120)
print(f"INPUT_PATH         : {INPUT_PATH}")
print(f"FULL_PANEL_PATH    : {FULL_PANEL_PATH}")
print(f"EDA_OUT_DIR        : {EDA_OUT_DIR}")
print(f"CLUSTER_INPUT_OUT  : {CLUSTER_INPUT_OUT}")
print(f"CLUSTER_META_DIR   : {CLUSTER_META_DIR}")
print(f"TRAIN_SCOPE        : {scope_name}")

# -------------------------
# 4) EXPORT CLUSTERING INPUT (long format, independent for later notebook)
# -------------------------
clustering_input = (
    df_train
    .select("pickup_bin_30m", "PULocationID", "pickup_demand")
    .orderBy("pickup_bin_30m", "PULocationID")
)

(
    clustering_input
    .write
    .mode("overwrite")
    .parquet(CLUSTER_INPUT_OUT)
)

cluster_input_summary = clustering_input.agg(
    F.count("*").alias("rows"),
    F.countDistinct("pickup_bin_30m").alias("n_time_bins"),
    F.countDistinct("PULocationID").alias("n_locations"),
    F.min("pickup_bin_30m").alias("min_time"),
    F.max("pickup_bin_30m").alias("max_time")
).first().asDict()

with open(os.path.join(CLUSTER_META_DIR, "clustering_input_summary.json"), "w", encoding="utf-8") as f:
    json.dump({k: (str(v) if not isinstance(v, (int, float, str, bool, type(None))) else v) for k, v in cluster_input_summary.items()}, f, indent=2, ensure_ascii=False)

# -------------------------
# 5) EDA STATS
# -------------------------
overall_summary = df_train.agg(
    F.count("*").alias("rows"),
    F.countDistinct("pickup_bin_30m").alias("n_time_bins"),
    F.countDistinct("PULocationID").alias("n_locations"),
    F.min("pickup_bin_30m").alias("min_time"),
    F.max("pickup_bin_30m").alias("max_time"),
    F.avg("pickup_demand").alias("avg_pickup_demand"),
    F.stddev("pickup_demand").alias("std_pickup_demand"),
    F.max("pickup_demand").alias("max_pickup_demand")
).first().asDict()

with open(os.path.join(EDA_OUT_DIR, "overall_summary.json"), "w", encoding="utf-8") as f:
    json.dump({k: (str(v) if not isinstance(v, (int, float, str, bool, type(None))) else v) for k, v in overall_summary.items()}, f, indent=2, ensure_ascii=False)

location_stats_pd = build_location_stats(df_train).orderBy(F.desc("active_ratio"), F.desc("avg_pickup_demand")).toPandas()

if os.path.exists(TAXI_ZONE_LOOKUP_CSV):
    lookup_pd = pd.read_csv(TAXI_ZONE_LOOKUP_CSV)
    if "LocationID" in lookup_pd.columns:
        lookup_pd = lookup_pd.rename(columns={"LocationID": "PULocationID"})
    location_stats_pd = location_stats_pd.merge(lookup_pd, on="PULocationID", how="left")

dense_candidates = location_stats_pd.sort_values(["active_ratio", "avg_pickup_demand"], ascending=[False, False]).head(TOP_N_DENSE).copy()
borderline_candidates = location_stats_pd.sort_values(["active_ratio", "avg_pickup_demand"], ascending=[True, False]).head(TOP_N_BORDERLINE).copy()

atomic_local_csv(location_stats_pd, os.path.join(EDA_OUT_DIR, "location_stats_train_scope.csv"))
atomic_local_csv(dense_candidates, os.path.join(EDA_OUT_DIR, "dense_locations_top.csv"))
atomic_local_csv(borderline_candidates, os.path.join(EDA_OUT_DIR, "borderline_locations_top.csv"))

# -------------------------
# 6) PROFILE EXPORTS
# -------------------------
hourly_profile = (
    df_train.groupBy("hour")
            .agg(F.avg("pickup_demand").alias("avg_pickup_demand"))
            .orderBy("hour")
            .toPandas()
)
hourly_profile.to_csv(os.path.join(EDA_OUT_DIR, "hourly_profile.csv"), index=False)

weekly_profile = (
    df_train.groupBy("day_of_week")
            .agg(F.avg("pickup_demand").alias("avg_pickup_demand"))
            .orderBy("day_of_week")
            .toPandas()
)
weekly_profile.to_csv(os.path.join(EDA_OUT_DIR, "weekly_profile.csv"), index=False)

dow_hour_profile = (
    df_train.groupBy("day_of_week", "hour")
            .agg(F.avg("pickup_demand").alias("avg_pickup_demand"))
            .orderBy("day_of_week", "hour")
            .toPandas()
)
dow_hour_profile.to_csv(os.path.join(EDA_OUT_DIR, "day_of_week_hour_profile.csv"), index=False)

# -------------------------
# 7) PLOTS
# -------------------------
plt.figure(figsize=(8, 5))
location_stats_pd["active_ratio"].dropna().hist(bins=40)
plt.title("Histogram of active_ratio")
plt.xlabel("active_ratio")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(EDA_OUT_DIR, "hist_active_ratio.png"), dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
location_stats_pd["avg_pickup_demand"].dropna().hist(bins=40)
plt.title("Histogram of avg_pickup_demand")
plt.xlabel("avg_pickup_demand")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(EDA_OUT_DIR, "hist_avg_pickup_demand.png"), dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
plt.scatter(location_stats_pd["active_ratio"], location_stats_pd["avg_pickup_demand"], s=10)
plt.title("active_ratio vs avg_pickup_demand")
plt.xlabel("active_ratio")
plt.ylabel("avg_pickup_demand")
plt.tight_layout()
plt.savefig(os.path.join(EDA_OUT_DIR, "scatter_active_ratio_vs_avg_demand.png"), dpi=150)
plt.close()

plt.figure(figsize=(9, 4))
plt.plot(hourly_profile["hour"], hourly_profile["avg_pickup_demand"])
plt.title("Hourly demand profile")
plt.xlabel("hour")
plt.ylabel("avg_pickup_demand")
plt.tight_layout()
plt.savefig(os.path.join(EDA_OUT_DIR, "hourly_profile.png"), dpi=150)
plt.close()

plt.figure(figsize=(9, 4))
plt.plot(weekly_profile["day_of_week"], weekly_profile["avg_pickup_demand"])
plt.title("Weekly demand profile")
plt.xlabel("day_of_week (0=Mon)")
plt.ylabel("avg_pickup_demand")
plt.tight_layout()
plt.savefig(os.path.join(EDA_OUT_DIR, "weekly_profile.png"), dpi=150)
plt.close()

pivot_heat = dow_hour_profile.pivot(index="day_of_week", columns="hour", values="avg_pickup_demand")
plt.figure(figsize=(12, 4))
plt.imshow(pivot_heat.values, aspect="auto")
plt.title("Day-of-week x hour average demand")
plt.xlabel("hour")
plt.ylabel("day_of_week (0=Mon)")
plt.colorbar()
plt.tight_layout()
plt.savefig(os.path.join(EDA_OUT_DIR, "day_of_week_hour_heatmap.png"), dpi=150)
plt.close()

# representative time-series plots
plot_loc_ids = pd.concat([dense_candidates[["PULocationID"]], borderline_candidates[["PULocationID"]]], axis=0)["PULocationID"].drop_duplicates().tolist()
for loc_id in plot_loc_ids:
    ts_pd = (
        df_train.filter(F.col("PULocationID") == int(loc_id))
                .select("pickup_bin_30m", "pickup_demand")
                .orderBy("pickup_bin_30m")
                .toPandas()
    )
    if len(ts_pd) == 0:
        continue
    ts_pd = downsample_pdf(ts_pd, MAX_PLOT_POINTS)
    plt.figure(figsize=(12, 3))
    plt.plot(pd.to_datetime(ts_pd["pickup_bin_30m"]), ts_pd["pickup_demand"])
    plt.title(f"Demand time series — PULocationID={int(loc_id)}")
    plt.xlabel("pickup_bin_30m")
    plt.ylabel("pickup_demand")
    plt.tight_layout()
    plt.savefig(os.path.join(EDA_OUT_DIR, f"timeseries_location_{int(loc_id)}.png"), dpi=150)
    plt.close()

# -------------------------
# 8) EXPORT MANIFEST
# -------------------------
export_manifest = {
    "input_path": INPUT_PATH,
    "train_scope": scope_name,
    "eda_out_dir": EDA_OUT_DIR,
    "clustering_input_out": CLUSTER_INPUT_OUT,
    "clustering_input_summary_json": os.path.join(CLUSTER_META_DIR, "clustering_input_summary.json"),
    "overall_summary_json": os.path.join(EDA_OUT_DIR, "overall_summary.json"),
    "location_stats_csv": os.path.join(EDA_OUT_DIR, "location_stats_train_scope.csv"),
    "dense_csv": os.path.join(EDA_OUT_DIR, "dense_locations_top.csv"),
    "borderline_csv": os.path.join(EDA_OUT_DIR, "borderline_locations_top.csv"),
    "hourly_profile_csv": os.path.join(EDA_OUT_DIR, "hourly_profile.csv"),
    "weekly_profile_csv": os.path.join(EDA_OUT_DIR, "weekly_profile.csv"),
    "day_of_week_hour_profile_csv": os.path.join(EDA_OUT_DIR, "day_of_week_hour_profile.csv")
}
with open(os.path.join(EDA_OUT_DIR, "export_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(export_manifest, f, indent=2, ensure_ascii=False)

# -------------------------
# 9) PRINT SUMMARY
# -------------------------
print("\n" + "=" * 120)
print("STEP 2 DONE")
print("=" * 120)
print("EDA + clustering input đã được xuất.")
print("Saved:")
for k, v in export_manifest.items():
    print(f" - {k}: {v}")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-799dba57-b37b-4d86-b7c6-ac31b3d3defd;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 114ms :: artifacts dl 4ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

STEP 2 — DEMAND EDA + EXPORT CLUSTERING INPUT
INPUT_PATH         : /user/data/preprocess/model_ready
FULL_PANEL_PATH    : /user/data/preprocess/full_panel
EDA_OUT_DIR        : /workspace/code/kshape/demand
CLUSTER_INPUT_OUT  : /user/data/clustering_input/train_series_long
CLUSTER_META_DIR   : /workspace/code/kshape/clustering_meta
TRAIN_SCOPE        : dataset_split=train



STEP 2 DONE
EDA + clustering input đã được xuất.
Saved:
 - input_path: /user/data/preprocess/model_ready
 - train_scope: dataset_split=train
 - eda_out_dir: /workspace/code/kshape/demand
 - clustering_input_out: /user/data/clustering_input/train_series_long
 - clustering_input_summary_json: /workspace/code/kshape/clustering_meta/clustering_input_summary.json
 - overall_summary_json: /workspace/code/kshape/demand/overall_summary.json
 - location_stats_csv: /workspace/code/kshape/demand/location_stats_train_scope.csv
 - dense_csv: /workspace/code/kshape/demand/dense_locations_top.csv
 - borderline_csv: /workspace/code/kshape/demand/borderline_locations_top.csv
 - hourly_profile_csv: /workspace/code/kshape/demand/hourly_profile.csv
 - weekly_profile_csv: /workspace/code/kshape/demand/weekly_profile.csv
 - day_of_week_hour_profile_csv: /workspace/code/kshape/demand/day_of_week_hour_profile.csv
